## 객실의 사용 여부 관련 데이터
1. 데이터를 로드(hotel_bookings.csv)
2. 데이터에 대한 정보를 확인
3. 해당 데이터에서 문제가 있는 부분을 확인하여 수정

In [1]:
import pandas as pd
import numpy as np

In [10]:
#  data 폴더 안에 hotel_bookings.csv 파일을 로드
hotel = pd.read_csv("../data/hotel_bookings.csv")
# 데이터를 로드하고 문제점(랜덤포레스트 학습)
hotel.head(2)

,is_canceled,deposit_type,lead_time,stays_in_weekend_nights,stays_in_week_nights,is_repeated_guest,previous_cancellations,previous_bookings_not_canceled,booking_changes,days_in_waiting_list,adr
0,0,No Deposit,105.0,2,5,NaN,0,0,1,0,131.50
1,0,No Deposit,303.0,2,2,NaN,0,0,0,0,73.95


In [12]:
# 데이터프레임의 정보 확인
hotel.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20000 entries, 0 to 19999
Data columns (total 11 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   is_canceled                     20000 non-null  int64  
 1   deposit_type                    20000 non-null  object 
 2   lead_time                       19995 non-null  float64
 3   stays_in_weekend_nights         20000 non-null  int64  
 4   stays_in_week_nights            20000 non-null  int64  
 5   is_repeated_guest               19642 non-null  float64
 6   previous_cancellations          20000 non-null  int64  
 7   previous_bookings_not_canceled  20000 non-null  int64  
 8   booking_changes                 20000 non-null  int64  
 9   days_in_waiting_list            20000 non-null  int64  
 10  adr                             18937 non-null  float64
dtypes: float64(3), int64(7), object(1)
memory usage: 1.7+ MB


In [6]:
hotel['deposit_type'].value_counts()

deposit_type
No Deposit    19138
Non Refund      834
Refundable       28
Name: count, dtype: int64

In [9]:
hotel.describe()

,is_canceled,lead_time,stays_in_weekend_nights,stays_in_week_nights,is_repeated_guest,previous_cancellations,previous_bookings_not_canceled,booking_changes,days_in_waiting_list,adr
count,20000.00000,19995.000000,20000.000000,20000.000000,19642.000000,20000.000000,20000.000000,20000.000000,20000.000000,18937.000000
mean,0.12000,85.978345,0.892550,2.380400,0.038133,0.032900,0.169050,0.269400,1.983950,101.410239
std,0.32497,96.427240,0.952077,1.777345,0.191521,0.455552,1.502426,0.687566,15.927212,49.245097
min,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,-6.380000
25%,0.00000,11.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,68.800000
50%,0.00000,51.000000,1.000000,2.000000,0.000000,0.000000,0.000000,0.000000,0.000000,94.500000
75%,0.00000,132.000000,2.000000,3.000000,0.000000,0.000000,0.000000,0.000000,0.000000,126.000000
max,1.00000,629.000000,13.000000,30.000000,1.000000,26.000000,66.000000,17.000000,379.000000,451.500000


### 해당 데이터의 컬럼의 의미
is_canceled : 예약 취소 여부 (0= 취소 안됨, 1=취소)

deposit_type : 보증금 유형(No Deposit, Not Refund, Refundable)

lead_time : 예약일과 실제 도착일 사이의 일수

stays_in_weekend_nights : 주말의 숙박 일수

stays_in_week_nights : 주중의 숙박 일수

is_repeated_guest : 재방문 고객 여부 (0=신규, 1=재방문)

previous_cancellations : 과거 예약 취소 횟수

previous_bookings_not_canceled : 과거 예약 중 취소되지 않은 건수

booking_changes : 예약 후 변경 횟수

days_in_waiting_list : 대기자가 있었던 일수

adr : 평균 일일 객실 요금

In [15]:
# 결측치가 존재하는 것은 확인 -> 실제 결측치의 개수를 확인
# 결측치가 존재 여부 함수 -> bool 타입의 데이터프레임이 생성
# bool의 데이터들을 합산하여 컬럼별로 확인
hotel.isna().sum()

is_canceled                          0
deposit_type                         0
lead_time                            5
stays_in_weekend_nights              0
stays_in_week_nights                 0
is_repeated_guest                  358
previous_cancellations               0
previous_bookings_not_canceled       0
booking_changes                      0
days_in_waiting_list                 0
adr                               1063
dtype: int64

In [18]:
# 결측치의 비율 -> 결측치의 개수 / 데이터프레임의 길이 * 100
print("lead_time 컬럼의 결측치의 비율 ", round(5 / len(hotel) * 100, 2))
print("is_repeated_guest 컬럼의 결측치의 비율 ", round(358 / len(hotel) * 100, 2))
print("adr 컬럼의 결측치의 비율 ", round(1063/ len(hotel) * 100, 2))

lead_time 컬럼의 결측치의 비율  0.03
is_repeated_guest 컬럼의 결측치의 비율  1.79
adr 컬럼의 결측치의 비율  5.32


- lead_time 컬럼의 결측치의 비율은 매우 작기 때문에 제거 -> 결측치인 인덱스를 제외
- is_repeated_guest는 해당 데이터에서 개수가 많은 데이터로 결측치를 채워준다.
- adr 컬럼의 결측치는 해당 데이터들을 확인하고 특정한 조건에 맞춰서 데이터를 채워준다.

In [22]:
# lead_time의 결측치가 존재하는 인덱스를 제외
# 제거한다(drop) + 결측치(na) -> 결측치가 존재하는 행이나 열을 제거하는 함수
hotel.dropna(subset = ['lead_time'], axis = 0, inplace = True)

In [ ]:
# 값들의 빈도수를 체크하는 함수를 사용
hotel['is_repeated_guest'].value_counts()

is_repeated_guest
0.0    18888
1.0      749
Name: count, dtype: int64

In [27]:
hotel['is_repeated_guest'].sum()

np.float64(749.0)

In [29]:
# is_repeated_guest 컬럼의 결측치는 0으로 채워준다.
hotel['is_repeated_guest'].fillna(0, inplace=True)

In [31]:
hotel.isna().sum()

is_canceled                          0
deposit_type                         0
lead_time                            0
stays_in_weekend_nights              0
stays_in_week_nights                 0
is_repeated_guest                    0
previous_cancellations               0
previous_bookings_not_canceled       0
booking_changes                      0
days_in_waiting_list                 0
adr                               1063
dtype: int64

In [34]:
hotel['adr'].describe()

count    18932.000000
mean       101.410702
std         49.241204
min         -6.380000
25%         68.822500
50%         94.500000
75%        126.000000
max        451.500000
Name: adr, dtype: float64

In [46]:
# 통계 정보를 확인하니 객실 요금 평균에 음수가 존재 -> 이상한 데이터가 발견
# 이상치 데이터는 제거
# 인덱스의 조건식을 생성 -> 객실 요금 데이터에서 0보다 작은
flag = hotel['adr'] < 0

hotel = hotel.loc[~flag]

In [58]:
# adr의 결측치들을 deposit_type의 값에 따라 그룹화를 하고 평균의 adr의 값을 채워준다.

# deposit_type에 따른 adr의 평균값을 확인
deposit_adr_mean = hotel.groupby(['deposit_type'])['adr'].mean().to_dict()

In [62]:
# hotel 복사본 생성
test_hotel = hotel.copy()

In [ ]:
null_flag = test_hotel['adr'].isna()
for key in deposit_adr_mean:
    # print(key)
    # print(deposit_adr_mean[key])
    # 인덱스 조건식 -> deposit_type이 key와 같다.
    flag = test_hotel['deposit_type'] == key
    # flag와 null_flag 두 조건식이 모두 만족하는 인덱스 필터
    # 첫번째 반복 시에는 deposit_type이 No Deposit이고 adr이 결측치인 데이터를 선택
    test_hotel.loc[flag & null_flag, 'adr'] = deposit_adr_mean[key]
    # break

In [70]:
test_hotel.isna().sum()

is_canceled                       0
deposit_type                      0
lead_time                         0
stays_in_weekend_nights           0
stays_in_week_nights              0
is_repeated_guest                 0
previous_cancellations            0
previous_bookings_not_canceled    0
booking_changes                   0
days_in_waiting_list              0
adr                               0
dtype: int64

In [74]:
# map(), apply()
hotel['adr'].map(
    lambda x : print(x)
)

131.5
73.95
nan
80.3
60.9
88.4
105.9
76.67
84.0
42.0
79.84
nan
88.0
126.0
nan
55.0
158.77
56.0
124.1
126.0
77.5
67.0
58.0
72.25
145.1
157.27
nan
100.0
89.0
170.0
67.0
nan
87.0
nan
90.0
107.1
123.0
85.85
56.1
30.24
62.0
110.88
0.0
nan
nan
42.3
0.0
107.1
283.67
55.0
69.0
105.0
62.0
88.0
77.18
134.14
25.11
150.3
168.3
115.2
108.15
130.0
96.0
48.0
75.0
104.5
111.0
130.0
118.13
98.1
168.3
98.0
74.13
82.24
196.0
58.0
87.0
60.0
144.0
118.8
0.0
nan
97.71
118.0
79.2
90.0
245.0
45.0
65.0
88.0
91.0
120.6
106.0
100.0
91.0
88.2
148.0
106.25
89.0
152.0
30.0
90.0
80.0
176.8
100.0
102.8
144.63
27.0
0.0
80.1
47.0
101.15
130.0
60.98
62.0
159.84
161.1
162.0
91.33
0.0
121.14
97.0
62.0
115.0
62.5
75.0
121.0
94.5
44.5
176.64
173.0
130.0
112.0
141.9
36.0
146.11
nan
90.0
126.0
62.0
100.8
54.6
42.0
82.0
169.0
77.85
50.0
35.0
95.0
177.0
89.4
149.4
73.8
0.0
152.0
90.95
48.0
87.0
119.7
nan
60.0
75.0
35.0
162.33
110.0
122.5
35.46
45.0
96.0
0.0
115.0
89.1
30.0
85.67
100.5
nan
82.35
153.62
164.0
120.0
38.0
170.57
75

0        None
1        None
2        None
3        None
4        None
         ... 
19995    None
19996    None
19997    None
19998    None
19999    None
Name: adr, Length: 19994, dtype: object

In [73]:
hotel['adr'].apply(
    lambda x : print(x)
)

131.5
73.95
nan
80.3
60.9
88.4
105.9
76.67
84.0
42.0
79.84
nan
88.0
126.0
nan
55.0
158.77
56.0
124.1
126.0
77.5
67.0
58.0
72.25
145.1
157.27
nan
100.0
89.0
170.0
67.0
nan
87.0
nan
90.0
107.1
123.0
85.85
56.1
30.24
62.0
110.88
0.0
nan
nan
42.3
0.0
107.1
283.67
55.0
69.0
105.0
62.0
88.0
77.18
134.14
25.11
150.3
168.3
115.2
108.15
130.0
96.0
48.0
75.0
104.5
111.0
130.0
118.13
98.1
168.3
98.0
74.13
82.24
196.0
58.0
87.0
60.0
144.0
118.8
0.0
nan
97.71
118.0
79.2
90.0
245.0
45.0
65.0
88.0
91.0
120.6
106.0
100.0
91.0
88.2
148.0
106.25
89.0
152.0
30.0
90.0
80.0
176.8
100.0
102.8
144.63
27.0
0.0
80.1
47.0
101.15
130.0
60.98
62.0
159.84
161.1
162.0
91.33
0.0
121.14
97.0
62.0
115.0
62.5
75.0
121.0
94.5
44.5
176.64
173.0
130.0
112.0
141.9
36.0
146.11
nan
90.0
126.0
62.0
100.8
54.6
42.0
82.0
169.0
77.85
50.0
35.0
95.0
177.0
89.4
149.4
73.8
0.0
152.0
90.95
48.0
87.0
119.7
nan
60.0
75.0
35.0
162.33
110.0
122.5
35.46
45.0
96.0
0.0
115.0
89.1
30.0
85.67
100.5
nan
82.35
153.62
164.0
120.0
38.0
170.57
75

0        None
1        None
2        None
3        None
4        None
         ... 
19995    None
19996    None
19997    None
19998    None
19999    None
Name: adr, Length: 19994, dtype: object

- 1차원 스리즈 데이터에서 map(), apply() 함수는 같은 행동을 한다.

In [ ]:
hotel.map(
    lambda x : print(x)
)
# DataFrame에서 map() 함수는 첫번째 스리즈의 values들을 모두 탐색하고 다음 스리즈의 value로 넘어간다.

0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0


,is_canceled,deposit_type,lead_time,stays_in_weekend_nights,stays_in_week_nights,is_repeated_guest,previous_cancellations,previous_bookings_not_canceled,booking_changes,days_in_waiting_list,adr
0,None,None,None,None,None,None,None,None,None,None,None
1,None,None,None,None,None,None,None,None,None,None,None
2,None,None,None,None,None,None,None,None,None,None,None
3,None,None,None,None,None,None,None,None,None,None,None
4,None,None,None,None,None,None,None,None,None,None,None
...,...,...,...,...,...,...,...,...,...,...,...
19995,None,None,None,None,None,None,None,None,None,None,None
19996,None,None,None,None,None,None,None,None,None,None,None
19997,None,None,None,None,None,None,None,None,None,None,None
19998,None,None,None,None,None,None,None,None,None,None,None


In [ ]:
hotel.apply(
    lambda x : print(x)
)
# DataFrame에서 apply() 함수는 스리즈 별로 탐색을 하는 함수
# 차원을 한개 축소하여 데이터를 확인

0        0
1        0
2        0
3        0
4        0
        ..
19995    1
19996    1
19997    1
19998    1
19999    1
Name: is_canceled, Length: 19994, dtype: int64
0        No Deposit
1        No Deposit
2        No Deposit
3        No Deposit
4        No Deposit
            ...    
19995    Non Refund
19996    Non Refund
19997    Non Refund
19998    No Deposit
19999    Non Refund
Name: deposit_type, Length: 19994, dtype: object
0        105.0
1        303.0
2         33.0
3         48.0
4        216.0
         ...  
19995     89.0
19996    101.0
19997    277.0
19998      0.0
19999     40.0
Name: lead_time, Length: 19994, dtype: float64
0        2
1        2
2        2
3        0
4        4
        ..
19995    2
19996    0
19997    1
19998    0
19999    0
Name: stays_in_weekend_nights, Length: 19994, dtype: int64
0        5
1        2
2        3
3        1
4        7
        ..
19995    2
19996    3
19997    2
19998    1
19999    2
Name: stays_in_week_nights, Length: 19994, dtype: 

is_canceled                       None
deposit_type                      None
lead_time                         None
stays_in_weekend_nights           None
stays_in_week_nights              None
is_repeated_guest                 None
previous_cancellations            None
previous_bookings_not_canceled    None
booking_changes                   None
days_in_waiting_list              None
adr                               None
dtype: object

In [80]:
# groupby() 함수와 apply()를 조합하여 deposit_type에 따라서 adr의 결측치를 평균값으로 대체
adj_hotel = hotel.groupby('deposit_type').apply(
    lambda x : x.fillna(x.mean())
)

In [ ]:
# adj_hotel의 인덱스 중 deposit_type 인덱스를 다시 value값으로 넘겨온다.
adj_hotel.info()

<class 'pandas.core.frame.DataFrame'>
MultiIndex: 19994 entries, ('No Deposit', np.int64(0)) to ('Refundable', np.int64(16851))
Data columns (total 10 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   is_canceled                     19994 non-null  int64  
 1   lead_time                       19994 non-null  float64
 2   stays_in_weekend_nights         19994 non-null  int64  
 3   stays_in_week_nights            19994 non-null  int64  
 4   is_repeated_guest               19994 non-null  float64
 5   previous_cancellations          19994 non-null  int64  
 6   previous_bookings_not_canceled  19994 non-null  int64  
 7   booking_changes                 19994 non-null  int64  
 8   days_in_waiting_list            19994 non-null  int64  
 9   adr                             19994 non-null  float64
dtypes: float64(3), int64(7)
memory usage: 2.2+ MB


In [87]:
# 멀티인덱스인 경우
# 특정 인덱스만 초기화하고 싶으면 reset_index()에서 level 매개변수 설정 변경
adj_hotel.reset_index(level='deposit_type', inplace=True)

In [89]:
# 종속 변수 -> 취소여부의 데이터 균형 부분 확인
adj_hotel['is_canceled'].value_counts()

is_canceled
0    17594
1     2400
Name: count, dtype: int64

- 종속변수 데이터의 균형 문제가 발생 -> 약 7:1 정도의 불균형이 발생
    - 원본의 데이터를 그대로 유지해서 불균형 문제를 학습이 모델이 처리하도록 실행
    - 오버샘플링을 이용하여 모델에 학습
        - 랜덤오버 샘플링
        - SMOTE

In [90]:
adj_hotel['deposit_type'].unique()

array(['No Deposit', 'Non Refund', 'Refundable'], dtype=object)

In [93]:
# deposit_type이 문자형 데이터이기 때문에 0, 1, 2로 데이터를 변경
adj_hotel['deposit_type'] = adj_hotel['deposit_type'].map(
    {
        'No Deposit' : 0,
        'Non Refund' : 1,
        'Refundable' : 2
    }
)

1. adj_hotel에서 독립 변수와 종속 변수로 데이터를 나눠준다.
2. train, test로 데이터를 분할 (7:3)
    - 종속 변수의 데이터의 비율로 데이터를 분할
3. 랜덤포레스트 분류 모델을 생성
4. train 데이터를 이용하여 학습
5. test 데이터를 이용하여 예측
6. 평가 지표 중 정확도를 확인

In [117]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
import time

In [123]:
# 시작 타이머 지정
start = time.time()
# 독립, 종속
x = adj_hotel.drop('is_canceled', axis=1).values
y = adj_hotel['is_canceled'].values
# train, test로 데이터 분할
X_train, X_test, Y_train, Y_test = train_test_split(
    x, y,
    test_size=0.3, stratify=y
)
# 모델 생성
clf = RandomForestClassifier()
# 모델 학습
clf.fit(X_train, Y_train)
# 모델 예측
pred = clf.predict(X_test)
end = time.time()
# 평가 지표
print("정확도 :", round(
    accuracy_score(Y_test, pred), 2
))
# 분류 보고서 출력
print(classification_report(Y_test, pred))
# 코드의 실행 시간 출력
print("코드 진행 시간 :", end-start)

정확도 : 0.92
              precision    recall  f1-score   support

           0       0.93      0.99      0.95      5279
           1       0.80      0.42      0.55       720

    accuracy                           0.92      5999
   macro avg       0.86      0.70      0.75      5999
weighted avg       0.91      0.92      0.91      5999

코드 진행 시간 : 1.4807195663452148


In [155]:
# 원데이터에서 랜덤 오버 샘플링 작업을 하여 샘플링이 된 데이터셋을 학습
# 독립 변수와 종속 변수를 이용하여 랜덤 오버 샘플링
from imblearn.over_sampling import RandomOverSampler
from imblearn.over_sampling import SMOTE

In [156]:
# 샘플러 생성
ros = RandomOverSampler(random_state=42)
smote = SMOTE(random_state=42)

In [157]:
x_over, y_over = ros.fit_resample(x, y)
x_sm, y_sm = smote.fit_resample(x, y)

In [158]:
# train test 데이터셋 분할
X_train_ros, X_test_ros, Y_train_ros, Y_test_ros = train_test_split(
    x_over, y_over,
    test_size=0.3, random_state=42,
    stratify=y_over
)
X_train_sm, X_test_sm, Y_train_sm, Y_test_sm = train_test_split(
    x_sm, y_sm,
    test_size=0.3, random_state=42,
    stratify=y_sm
)

In [159]:
# Random Over Sampling을 한 데이터를 이용하여 랜덤포레스트의 학습, 검증
start = time.time()
clf_ros = RandomForestClassifier()
clf_ros.fit(X_train_ros, Y_train_ros)
pred_ros = clf_ros.predict(X_test_ros)
end = time.time()
print(classification_report(Y_test_ros, pred_ros))
print("RandomOver 데이터의 소요 시간 : ", end-start)

              precision    recall  f1-score   support

           0       0.99      0.94      0.96      5279
           1       0.94      0.99      0.97      5278

    accuracy                           0.96     10557
   macro avg       0.97      0.96      0.96     10557
weighted avg       0.97      0.96      0.96     10557

RandomOver 데이터의 소요 시간 :  2.331523895263672


In [160]:
# SMOTE을 이용하여 랜덤포레스트에 학습, 검증
start = time.time()
clf_sm = RandomForestClassifier()
clf_sm.fit(X_train_sm, Y_train_sm)
pred_sm = clf_sm.predict(X_test_sm)
end = time.time()
print(classification_report(Y_test_sm, pred_sm))
print("SMOTE 데이터의 소요 시간 : ", end-start)

              precision    recall  f1-score   support

           0       0.92      0.98      0.95      5279
           1       0.98      0.92      0.95      5278

    accuracy                           0.95     10557
   macro avg       0.95      0.95      0.95     10557
weighted avg       0.95      0.95      0.95     10557

SMOTE 데이터의 소요 시간 :  2.6983907222747803
